# Embedding-dimension selection validation for ordinal HCFIM

This diagnostic notebook evaluates whether retaining ordinal dimension $m=4$ can be justified without using whole-cohort diagnostic labels. It separates three questions: label-free feasibility and stability, multiplicity-aware coordinate testing across $m=3,4,5$, and leakage-safe predictive comparison with $m$ either fixed in advance or selected only inside each outer training set.

## tl;dr

- In the post hoc temporal geometries, median ordinal-state occupancy for $m=5$ was **0.167** in alpha and **0.333** in broadband; nominal sample-to-state ratio alone therefore did not decide between $m=4$ and $m=5$.
- Median split-half Spearman stability across H/C/FIM was **0.879, 0.856, and 0.824** for $m=3,4,5$ in temporal alpha, and **0.791, 0.789, and 0.784** in temporal broadband.
- Control-versus-AD mean repeated nested ROC AUC was **0.444**, **0.742**, and **0.821** for fixed $m=3,4,5$; inner-fold selection yielded **0.789**.
- Control-versus-FTD mean repeated nested ROC AUC was **0.662**, **0.684**, and **0.689** for fixed $m=3,4,5$; inner-fold selection yielded **0.675**.
- Inner-fold preferences were not treated as proof of a universal dimension (control vs alzheimer: m=5 in 80% of outer folds; control vs frontotemporal dementia: m=5 in 44% of outer folds). Multiplicity-aware significant-coordinate counts were: control vs alzheimer m=3: 0/125; control vs alzheimer m=4: 3/125; control vs alzheimer m=5: 3/125; control vs frontotemporal dementia m=3: 0/125; control vs frontotemporal dementia m=4: 0/125; control vs frontotemporal dementia m=5: 0/125.
- The defensible manuscript correction is to remove the claim that $m=4$ was selected because it ranked highest in the full cohort. $m=4$ may remain an exploratory intermediate 24-state visualization, while any adaptive selection must occur inside training folds and requires external validation.


## Context & Methods

### Key assumptions

- The label-free audit uses the first 60 valid 10 s epochs per participant so duration is balanced.
- Occupancy and split-half stability are evaluated for the two manuscript-highlighted temporal geometries only; because those geometries were selected post hoc, this audit diagnoses estimator behavior but is not independent confirmation of those locations.
- Coordinate-wise inference covers all 25 regional band combinations and all three ordinal dimensions. H and FIM are tested once per region/band; C remains distance-specific.
- Predictive comparisons use all regional ordinal-Shannon features for the stated $m$, fixed $k=30$, identical outer partitions, and fold-local imputation, scaling, selection, and model fitting.
- The adaptive analysis selects $m$ and tree depth exclusively inside each outer training partition. Outer-fold labels are never used for this choice.
- Across-repetition percentile ranges describe partition sensitivity, not external-validation confidence intervals.

In [ ]:
from __future__ import annotations

import itertools
import json
import math
import os
import platform
import sys
import time
import warnings
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from IPython.display import Markdown, display
from joblib import Parallel, delayed
from scipy import signal, stats
from scipy.io import loadmat
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=RuntimeWarning)

SEED = 17536
N_REPEATS = 10
OUTER_FOLDS = 5
INNER_FOLDS = 3
N_FIXED_EPOCHS = 60
K_FEATURES = 30
N_TREES = 100
PARALLEL_JOBS = min(4, os.cpu_count() or 1)
M_VALUES = (3, 4, 5)
DEPTHS = (2, 3, None)

def resolve_project_root() -> Path:
    """Locate the repository without relying on a machine-specific path."""
    env_root = os.environ.get("HCFIM_PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
        raise FileNotFoundError(
            "HCFIM_PROJECT_ROOT does not point to the HCFIM-eeg-research repository."
        )

    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository. Run the notebook from within the repository "
        "or set HCFIM_PROJECT_ROOT."
    )

PROJECT_ROOT = resolve_project_root()
def resolve_dataset_root(project_root: Path) -> Path:
    """Locate ds004504 derivatives or use HCFIM_DATASET_ROOT."""
    candidates = []
    env_root = os.environ.get("HCFIM_DATASET_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([
        project_root / "data" / "ds004504",
        project_root / "data" / "data-08-00095",
        project_root / "data",
    ])
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "participants.tsv").is_file() and (candidate / "derivatives").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate ds004504. Set HCFIM_DATASET_ROOT to the dataset directory "
        "containing participants.tsv and derivatives/."
    )

DATASET_ROOT = resolve_dataset_root(PROJECT_ROOT)
RESULTS = PROJECT_ROOT / "results"
SRC = PROJECT_ROOT / "src"
OUT = Path(os.environ.get(
    "HCFIM_NOTEBOOK08_OUT",
    str(RESULTS / "embedding_dimension_selection_validation"),
))
FIGURES = OUT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import itm_geometry as geom

sns.set_theme(style="whitegrid", context="notebook")
COLORS = {3: "#315B7D", 4: "#D18B32", 5: "#8A5A9E", "adaptive": "#6F7D3C"}

print(f"Python: {platform.python_version()} | scikit-learn: {sklearn.__version__}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Output: {OUT}")

## Data

### 1. Load and validate participant-level inputs

In [ ]:
subject_features = pd.read_csv(RESULTS / "subject_features_full.csv")
participants = pd.read_csv(RESULTS / "participants_index.csv")
analysis_frame = subject_features.merge(
    participants[["participant_id", "diagnosis", "set_path", "set_exists"]],
    on=["participant_id", "diagnosis"], how="left", validate="one_to_one",
)

assert len(analysis_frame) == 88
assert analysis_frame["participant_id"].is_unique
assert analysis_frame["set_exists"].all()
assert analysis_frame["diagnosis"].value_counts().to_dict() == {
    "alzheimer": 36, "control": 29, "frontotemporal_dementia": 23,
}

feature_columns_by_m = {
    m: [column for column in subject_features.columns
        if column.startswith(f"hcfim__ordinal_shannon_m{m}_t1__")]
    for m in M_VALUES
}
assert {m: len(columns) for m, columns in feature_columns_by_m.items()} == {3: 1350, 4: 1350, 5: 1350}

inventory = pd.DataFrame({
    "item": ["Participants", "AD", "FTD", "Controls", "Ordinal features per m", "Raw derivatives"],
    "value": [88, 36, 23, 29, 1350, int(analysis_frame["set_exists"].sum())],
})
display(inventory)

## Results

### 2. Audit ordinal occupancy without diagnostic labels

The calculation below reconstructs the temporal regional waveform and evaluates the two manuscript-highlighted bands using balanced first-60-epoch data. Diagnostic labels are retained only as provenance and are not used to rank or choose $m$.

In [ ]:
CHANNEL_RENAME = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8"}
TEMPORAL_CHANNELS = ["T7", "T8", "P7", "P8"]
BANDS = {"temporal alpha": (8.0, 13.0), "temporal broadband": (0.5, 40.0)}

def load_set_matrix(set_path: Path) -> tuple[np.ndarray, float, list[str]]:
    payload = loadmat(set_path, simplify_cells=True)
    data = np.asarray(payload["data"], dtype=float)
    sfreq = float(payload["srate"])
    channel_locations = payload["chanlocs"]
    if isinstance(channel_locations, dict):
        channel_locations = [channel_locations]
    labels = [CHANNEL_RENAME.get(str(item["labels"]), str(item["labels"])) for item in channel_locations]
    return data, sfreq, labels

def epoch_slices(n_samples: int, sfreq: float, n_epochs: int = N_FIXED_EPOCHS):
    window = int(round(10.0 * sfreq))
    step = int(round(window * 0.5))
    for epoch_id, start in enumerate(range(0, n_samples - window + 1, step)):
        if epoch_id >= n_epochs:
            break
        yield epoch_id, start, start + window

def bandpass_window(values: np.ndarray, sfreq: float, band: tuple[float, float]) -> np.ndarray:
    values = signal.detrend(np.asarray(values, float), type="constant")
    nyquist = sfreq / 2.0
    low, high = band
    sos = signal.butter(
        4,
        [max(low, 0.01) / nyquist, min(high, nyquist - 0.5) / nyquist],
        btype="bandpass", output="sos",
    )
    return signal.sosfiltfilt(sos, values)

@lru_cache(maxsize=None)
def ordinal_lookup(m: int):
    permutations = np.asarray(list(itertools.permutations(range(m))), dtype=int)
    powers = (m ** np.arange(m)).astype(int)
    codes = permutations @ powers
    lookup = np.full(m ** m, -1, dtype=int)
    lookup[codes] = np.arange(len(permutations))
    return powers, lookup

def ordinal_counts(values: np.ndarray, m: int, tau: int = 1) -> np.ndarray:
    values = np.asarray(values, float)
    n_vectors = values.size - (m - 1) * tau
    embedded = np.column_stack([values[offset * tau:offset * tau + n_vectors] for offset in range(m)])
    ordering = np.argsort(embedded, axis=1, kind="stable")
    powers, lookup = ordinal_lookup(m)
    indices = lookup[ordering @ powers]
    assert np.all(indices >= 0)
    return np.bincount(indices, minlength=math.factorial(m))

def audit_subject(row) -> list[dict]:
    data, sfreq, labels = load_set_matrix(Path(row.set_path))
    channel_index = {label: index for index, label in enumerate(labels)}
    indices = [channel_index[channel] for channel in TEMPORAL_CHANNELS if channel in channel_index]
    if not indices:
        raise ValueError(f"No temporal channels found for {row.participant_id}")
    regional = np.nanmean(data[indices, :], axis=0)
    records = []
    for epoch_id, start, end in epoch_slices(regional.size, sfreq):
        window = regional[start:end]
        for configuration, band in BANDS.items():
            filtered = bandpass_window(window, sfreq, band)
            for m in M_VALUES:
                counts = ordinal_counts(filtered, m=m, tau=1)
                probabilities = counts / counts.sum()
                spec = geom.EntropySpec(
                    entropy_id=f"ordinal_shannon_m{m}_t1",
                    label=f"m={m}, tau=1", family="ordinal",
                    probability_domain="Bandt-Pompe ordinal patterns",
                    n_states=math.factorial(m), entropy_kind="shannon", m=m, tau=1,
                )
                point = geom.hcfim_point(
                    probabilities, spec,
                    distance_code=geom.DISTANCES["euclidean"]["code"],
                )
                positive = counts[counts > 0]
                records.append({
                    "participant_id": row.participant_id,
                    "diagnosis_provenance_only": row.diagnosis,
                    "configuration": configuration,
                    "epoch_id": epoch_id,
                    "m": m,
                    "n_vectors": int(counts.sum()),
                    "n_states": int(counts.size),
                    "observed_states": int((counts > 0).sum()),
                    "occupancy_fraction": float((counts > 0).mean()),
                    "zero_state_fraction": float((counts == 0).mean()),
                    "minimum_positive_count": int(positive.min()),
                    "median_positive_count": float(np.median(positive)),
                    "maximum_pattern_share": float(probabilities.max()),
                    "monotonic_pattern_share": float(probabilities[0] + probabilities[-1]),
                    "effective_state_fraction": float(np.exp(-np.sum(probabilities[probabilities > 0] * np.log(probabilities[probabilities > 0]))) / counts.size),
                    "H": point["H"], "C": point["C"], "FIM": point["FIM"],
                })
    return records

raw_start = time.time()
audits = Parallel(n_jobs=PARALLEL_JOBS, prefer="threads", verbose=5)(
    delayed(audit_subject)(row) for row in analysis_frame.itertuples(index=False)
)
epoch_audit = pd.DataFrame([record for subject_records in audits for record in subject_records])
assert len(epoch_audit) == 88 * N_FIXED_EPOCHS * len(BANDS) * len(M_VALUES)
epoch_audit.to_csv(OUT / "label_free_epoch_occupancy.csv", index=False)
print(f"Label-free raw audit completed in {(time.time() - raw_start) / 60:.1f} min; rows={len(epoch_audit):,}")

### 3. Summarize state occupancy and monotonic-pattern concentration

In [ ]:
occupancy_summary = (
    epoch_audit.groupby(["configuration", "m"], as_index=False)
    .agg(
        epochs=("epoch_id", "size"),
        occupancy_median=("occupancy_fraction", "median"),
        occupancy_p2_5=("occupancy_fraction", lambda x: np.percentile(x, 2.5)),
        occupancy_p97_5=("occupancy_fraction", lambda x: np.percentile(x, 97.5)),
        zero_state_median=("zero_state_fraction", "median"),
        min_positive_count_median=("minimum_positive_count", "median"),
        effective_state_fraction_median=("effective_state_fraction", "median"),
        maximum_pattern_share_median=("maximum_pattern_share", "median"),
        monotonic_share_median=("monotonic_pattern_share", "median"),
    )
)
occupancy_summary.to_csv(OUT / "label_free_occupancy_summary.csv", index=False)
display(occupancy_summary.style.format(precision=3))

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.8), sharey=False)
for ax, metric, ylabel in [
    (axes[0], "occupancy_fraction", "Observed ordinal states / m!"),
    (axes[1], "monotonic_pattern_share", "Ascending + descending pattern share"),
]:
    sns.boxplot(
        data=epoch_audit, x="m", y=metric, hue="configuration",
        showfliers=False, palette=["#315B7D", "#D18B32"], ax=ax,
    )
    ax.set(xlabel="Embedding dimension m", ylabel=ylabel)
    ax.legend(frameon=False, title="Configuration", fontsize=8)
fig.suptitle("Label-free ordinal-domain feasibility in balanced first-60-epoch data")
fig.tight_layout()
fig.savefig(FIGURES / "label_free_occupancy_and_monotonic_share.png", dpi=220, bbox_inches="tight")
plt.show()

### 4. Quantify split-half stability without labels

Participant summaries from epochs 0-29 and 30-59 are compared across participants. Spearman correlation measures rank stability; the relative mean absolute difference is scaled by the pooled between-participant standard deviation.

In [ ]:
half_frame = epoch_audit.copy()
half_frame["half"] = np.where(half_frame["epoch_id"] < N_FIXED_EPOCHS / 2, "first", "second")
half_means = (
    half_frame.groupby(["participant_id", "configuration", "m", "half"], as_index=False)[["H", "C", "FIM"]]
    .mean()
)

stability_rows = []
for (configuration, m), group in half_means.groupby(["configuration", "m"]):
    wide = group.pivot(index="participant_id", columns="half", values=["H", "C", "FIM"])
    for coordinate in ["H", "C", "FIM"]:
        first = wide[(coordinate, "first")].to_numpy()
        second = wide[(coordinate, "second")].to_numpy()
        rho = stats.spearmanr(first, second).statistic
        pooled_sd = np.std(np.concatenate([first, second]), ddof=1)
        stability_rows.append({
            "configuration": configuration, "m": m, "coordinate": coordinate,
            "spearman_rho": rho,
            "mean_absolute_difference": np.mean(np.abs(first - second)),
            "relative_mean_absolute_difference": np.mean(np.abs(first - second)) / pooled_sd if pooled_sd else np.nan,
        })

split_half_stability = pd.DataFrame(stability_rows)
split_half_stability.to_csv(OUT / "label_free_split_half_stability.csv", index=False)
display(split_half_stability.style.format(precision=3))

heatmap = split_half_stability.pivot_table(
    index=["configuration", "coordinate"], columns="m", values="spearman_rho"
)
fig, ax = plt.subplots(figsize=(7.2, 4.8))
sns.heatmap(heatmap, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax)
ax.set(title="Label-free split-half rank stability", xlabel="Embedding dimension m", ylabel="Configuration and coordinate")
fig.tight_layout()
fig.savefig(FIGURES / "label_free_split_half_stability.png", dpi=220, bbox_inches="tight")
plt.show()

### 5. Repeat coordinate-wise tests with $m$ included in the multiplicity family

The family contains $3	imes25	imes(1_H+1_{FIM}+3_C)=375$ tests per diagnostic comparison. This is a focused ordinal-Shannon audit, not a replacement for the broader 875-test analysis across all transformations.

In [ ]:
REGIONS = ("frontal", "temporal", "central", "parietal", "occipital")
BAND_NAMES = ("delta", "theta", "alpha", "beta", "broadband")
DISTANCES = ("euclidean", "jensen_shannon", "wootters")

def bh_adjust(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, float)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    output = np.empty_like(adjusted)
    output[order] = np.clip(adjusted, 0, 1)
    return output

def coordinate_tests(disease: str) -> pd.DataFrame:
    subset = subject_features.loc[subject_features["diagnosis"].isin(["control", disease])]
    records = []
    for m in M_VALUES:
        for region in REGIONS:
            for band in BAND_NAMES:
                specifications = [("H", "euclidean"), ("FIM", "euclidean")]
                specifications.extend(("C", distance) for distance in DISTANCES)
                for coordinate, distance in specifications:
                    column = f"hcfim__ordinal_shannon_m{m}_t1__{distance}__{region}__{band}__{coordinate}__mean"
                    assert column in subset.columns, column
                    control = subset.loc[subset["diagnosis"] == "control", column].to_numpy(float)
                    patient = subset.loc[subset["diagnosis"] == disease, column].to_numpy(float)
                    test = stats.mannwhitneyu(control, patient, alternative="two-sided")
                    records.append({
                        "comparison": f"control_vs_{disease}", "m": m, "region": region, "band": band,
                        "coordinate": coordinate, "distance": distance, "column": column,
                        "control_median": np.median(control), "patient_median": np.median(patient),
                        "median_difference_patient_minus_control": np.median(patient) - np.median(control),
                        "u_statistic": test.statistic, "p_value": test.pvalue,
                    })
    frame = pd.DataFrame(records)
    assert len(frame) == 375
    frame["q_value"] = bh_adjust(frame["p_value"].to_numpy())
    frame["fdr_0_05"] = frame["q_value"] <= 0.05
    return frame

coordinate_fdr = pd.concat([
    coordinate_tests("alzheimer"), coordinate_tests("frontotemporal_dementia")
], ignore_index=True)
coordinate_fdr.to_csv(OUT / "ordinal_m3_m4_m5_coordinate_fdr.csv", index=False)

fdr_summary = (
    coordinate_fdr.groupby(["comparison", "m"], as_index=False)
    .agg(tests=("p_value", "size"), significant=("fdr_0_05", "sum"), minimum_q=("q_value", "min"))
)
fdr_summary.to_csv(OUT / "ordinal_m3_m4_m5_fdr_summary.csv", index=False)
display(fdr_summary.style.format({"minimum_q": "{:.4g}"}))

significant_coordinates = coordinate_fdr.loc[coordinate_fdr["fdr_0_05"]].sort_values(["comparison", "q_value"])
significant_coordinates.to_csv(OUT / "ordinal_m3_m4_m5_fdr_significant.csv", index=False)
display(significant_coordinates.head(30).style.format(precision=4))

### 6. Define leakage-safe repeated nested comparisons

In [ ]:
TASKS = {
    "control_vs_alzheimer": ("control", "alzheimer"),
    "control_vs_frontotemporal_dementia": ("control", "frontotemporal_dementia"),
}

def make_pipeline(k: int, seed: int, depth) -> Pipeline:
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("select", SelectKBest(f_classif, k=k)),
        ("model", ExtraTreesClassifier(
            n_estimators=N_TREES, max_depth=depth, class_weight="balanced",
            random_state=seed, n_jobs=1,
        )),
    ])

def choose_depth(X, y, train_idx, seed):
    inner = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=seed + 100)
    candidates = []
    for depth in DEPTHS:
        scores = []
        for inner_fold, (inner_train, inner_test) in enumerate(inner.split(X[train_idx], y[train_idx]), start=1):
            estimator = make_pipeline(K_FEATURES, seed + inner_fold, depth)
            estimator.fit(X[train_idx][inner_train], y[train_idx][inner_train])
            probability = estimator.predict_proba(X[train_idx][inner_test])[:, 1]
            scores.append(roc_auc_score(y[train_idx][inner_test], probability))
        candidates.append((np.mean(scores), depth))
    candidates.sort(key=lambda item: (item[0], -1 if item[1] is None else -item[1]), reverse=True)
    return candidates[0][1], candidates[0][0]

def run_fixed_configuration(task_name: str, m: int):
    negative_label, positive_label = TASKS[task_name]
    frame = subject_features.loc[subject_features["diagnosis"].isin([negative_label, positive_label])].reset_index(drop=True)
    y = (frame["diagnosis"] == positive_label).astype(int).to_numpy()
    X = frame[feature_columns_by_m[m]].to_numpy(float)
    repeat_rows, fold_rows, oof_rows = [], [], []
    for repeat in range(N_REPEATS):
        outer_seed = SEED + repeat * 1000
        outer = StratifiedKFold(OUTER_FOLDS, shuffle=True, random_state=outer_seed)
        probability = np.full(len(y), np.nan)
        prediction = np.full(len(y), -1)
        for fold, (train_idx, test_idx) in enumerate(outer.split(X, y), start=1):
            fit_seed = outer_seed + fold
            depth, inner_auc = choose_depth(X, y, train_idx, fit_seed)
            estimator = make_pipeline(K_FEATURES, fit_seed, depth)
            estimator.fit(X[train_idx], y[train_idx])
            probability[test_idx] = estimator.predict_proba(X[test_idx])[:, 1]
            prediction[test_idx] = estimator.predict(X[test_idx])
            fold_rows.append({
                "task": task_name, "method": f"fixed_m{m}", "m": m,
                "repeat": repeat, "fold": fold, "selected_depth": str(depth), "inner_auc": inner_auc,
            })
        repeat_rows.append({
            "task": task_name, "method": f"fixed_m{m}", "m": m, "repeat": repeat,
            "roc_auc": roc_auc_score(y, probability),
            "balanced_accuracy": balanced_accuracy_score(y, prediction),
        })
        for index, row in frame.iterrows():
            oof_rows.append({
                "participant_id": row.participant_id, "diagnosis": row.diagnosis,
                "task": task_name, "method": f"fixed_m{m}", "m": m, "repeat": repeat,
                "y_true": y[index], "probability_positive": probability[index], "y_pred": prediction[index],
            })
    return repeat_rows, fold_rows, oof_rows

def run_adaptive_configuration(task_name: str):
    negative_label, positive_label = TASKS[task_name]
    frame = subject_features.loc[subject_features["diagnosis"].isin([negative_label, positive_label])].reset_index(drop=True)
    y = (frame["diagnosis"] == positive_label).astype(int).to_numpy()
    matrices = {m: frame[feature_columns_by_m[m]].to_numpy(float) for m in M_VALUES}
    repeat_rows, fold_rows, oof_rows = [], [], []
    for repeat in range(N_REPEATS):
        outer_seed = SEED + repeat * 1000
        outer = StratifiedKFold(OUTER_FOLDS, shuffle=True, random_state=outer_seed)
        probability = np.full(len(y), np.nan)
        prediction = np.full(len(y), -1)
        for fold, (train_idx, test_idx) in enumerate(outer.split(matrices[3], y), start=1):
            fit_seed = outer_seed + fold
            candidates = []
            for m in M_VALUES:
                depth, inner_auc = choose_depth(matrices[m], y, train_idx, fit_seed + m * 10_000)
                candidates.append((inner_auc, m, depth))
            candidates.sort(key=lambda item: (item[0], -item[1]), reverse=True)
            inner_auc, selected_m, depth = candidates[0]
            estimator = make_pipeline(K_FEATURES, fit_seed, depth)
            estimator.fit(matrices[selected_m][train_idx], y[train_idx])
            probability[test_idx] = estimator.predict_proba(matrices[selected_m][test_idx])[:, 1]
            prediction[test_idx] = estimator.predict(matrices[selected_m][test_idx])
            fold_rows.append({
                "task": task_name, "method": "inner_selected_m", "m": selected_m,
                "repeat": repeat, "fold": fold, "selected_depth": str(depth), "inner_auc": inner_auc,
            })
        repeat_rows.append({
            "task": task_name, "method": "inner_selected_m", "m": np.nan, "repeat": repeat,
            "roc_auc": roc_auc_score(y, probability),
            "balanced_accuracy": balanced_accuracy_score(y, prediction),
        })
        for index, row in frame.iterrows():
            oof_rows.append({
                "participant_id": row.participant_id, "diagnosis": row.diagnosis,
                "task": task_name, "method": "inner_selected_m", "m": np.nan, "repeat": repeat,
                "y_true": y[index], "probability_positive": probability[index], "y_pred": prediction[index],
            })
    return repeat_rows, fold_rows, oof_rows

### 7. Run fixed-$m$ and inner-selected-$m$ experiments

In [ ]:
classification_start = time.time()
fixed_jobs = [(task_name, m) for task_name in TASKS for m in M_VALUES]
fixed_outputs = Parallel(n_jobs=PARALLEL_JOBS, prefer="threads", verbose=5)(
    delayed(run_fixed_configuration)(task_name, m) for task_name, m in fixed_jobs
)
adaptive_outputs = Parallel(n_jobs=min(2, PARALLEL_JOBS), prefer="threads", verbose=5)(
    delayed(run_adaptive_configuration)(task_name) for task_name in TASKS
)
all_outputs = fixed_outputs + adaptive_outputs

classification_repeats = pd.DataFrame([row for output in all_outputs for row in output[0]])
classification_folds = pd.DataFrame([row for output in all_outputs for row in output[1]])
classification_oof = pd.DataFrame([row for output in all_outputs for row in output[2]])

classification_repeats.to_csv(OUT / "repeated_nested_embedding_dimension_metrics.csv", index=False)
classification_folds.to_csv(OUT / "repeated_nested_embedding_dimension_folds.csv", index=False)
classification_oof.to_csv(OUT / "repeated_nested_embedding_dimension_oof.csv", index=False)
print(f"Classification completed in {(time.time() - classification_start) / 60:.1f} min")

### 8. Summarize predictive sensitivity and inner-fold choices

In [ ]:
classification_summary = (
    classification_repeats.groupby(["task", "method"], as_index=False)
    .agg(
        repetitions=("repeat", "size"),
        auc_mean=("roc_auc", "mean"),
        auc_median=("roc_auc", "median"),
        auc_sd=("roc_auc", "std"),
        auc_p2_5=("roc_auc", lambda x: np.percentile(x, 2.5)),
        auc_p97_5=("roc_auc", lambda x: np.percentile(x, 97.5)),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
    )
)
classification_summary.to_csv(OUT / "repeated_nested_embedding_dimension_summary.csv", index=False)
display(classification_summary.style.format(precision=3))

selection_frequency = (
    classification_folds.loc[classification_folds["method"] == "inner_selected_m"]
    .groupby(["task", "m"], as_index=False)
    .agg(selected_outer_folds=("fold", "size"))
)
selection_frequency["selection_fraction"] = selection_frequency.groupby("task")["selected_outer_folds"].transform(lambda x: x / x.sum())
selection_frequency.to_csv(OUT / "inner_fold_m_selection_frequency.csv", index=False)
display(selection_frequency.style.format({"selection_fraction": "{:.1%}"}))

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.8), sharey=True)
method_order = ["fixed_m3", "fixed_m4", "fixed_m5", "inner_selected_m"]
palette = [COLORS[3], COLORS[4], COLORS[5], COLORS["adaptive"]]
for ax, task_name in zip(axes, TASKS):
    plot = classification_repeats.loc[classification_repeats["task"] == task_name]
    sns.boxplot(data=plot, x="method", y="roc_auc", order=method_order, palette=palette, showfliers=False, ax=ax)
    sns.stripplot(data=plot, x="method", y="roc_auc", order=method_order, color="#222222", size=4, jitter=0.12, ax=ax)
    ax.axhline(0.5, color="#666666", linestyle="--", linewidth=1)
    ax.set(title=task_name.replace("_", " "), xlabel="Embedding-dimension strategy", ylabel="Out-of-fold ROC AUC")
    ax.tick_params(axis="x", rotation=20)
fig.suptitle("Repeated nested performance with fixed or inner-selected embedding dimension")
fig.tight_layout()
fig.savefig(FIGURES / "repeated_nested_auc_by_embedding_dimension.png", dpi=220, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8.0, 4.5))
sns.barplot(data=selection_frequency, x="task", y="selection_fraction", hue="m", palette=[COLORS[3], COLORS[4], COLORS[5]], ax=ax)
ax.set(title="Embedding dimension selected inside outer training partitions", xlabel="Task", ylabel="Fraction of outer folds", ylim=(0, 1))
ax.tick_params(axis="x", rotation=15)
ax.legend(title="m", frameon=False)
fig.tight_layout()
fig.savefig(FIGURES / "inner_fold_m_selection_frequency.png", dpi=220, bbox_inches="tight")
plt.show()

# Manuscript-ready Table H2: selection frequencies belong to the adaptive strategy,
# not to the fixed-m rows.
frequency_text = {}
for task_name, group in selection_frequency.groupby("task"):
    parts = [f"m={int(row.m)}: {row.selection_fraction:.0%}" for row in group.itertuples(index=False)]
    frequency_text[task_name] = "; ".join(parts)

table_h2 = classification_summary[[
    "task", "method", "auc_mean", "auc_p2_5", "auc_p97_5"
]].copy()
table_h2["inner_selection_frequency"] = "n/a"
adaptive_mask = table_h2["method"].eq("inner_selected_m")
table_h2.loc[adaptive_mask, "inner_selection_frequency"] = table_h2.loc[
    adaptive_mask, "task"
].map(frequency_text)
table_h2["interpretation"] = ""
table_h2.loc[
    (table_h2["task"] == "control_vs_alzheimer")
    & (table_h2["method"] == "fixed_m3"),
    "interpretation",
] = "Across-repetition range is predominantly below 0.50; treated as unstable anti-learning, not useful discrimination."
table_h2.to_csv(OUT / "table_h2_corrected.csv", index=False)
display(table_h2.style.format({
    "auc_mean": "{:.3f}", "auc_p2_5": "{:.3f}", "auc_p97_5": "{:.3f}"
}))


## Takeaways

In [ ]:
label_free_overview = (
    occupancy_summary.merge(
        split_half_stability.groupby(["configuration", "m"], as_index=False)
        .agg(median_split_half_rho=("spearman_rho", "median"), median_relative_mae=("relative_mean_absolute_difference", "median")),
        on=["configuration", "m"], how="left",
    )
)
label_free_overview.to_csv(OUT / "label_free_embedding_dimension_overview.csv", index=False)

display(Markdown(
    "1. **No label-free result is interpreted as proving a unique optimal m.** Occupancy and split-half stability "
    "describe estimator feasibility in the two post hoc temporal geometries.\n"
    "2. **The multiplicity audit treats m as part of the tested family.** Any surviving coordinate is reported "
    "within a 375-test ordinal-Shannon family per comparison.\n"
    "3. **Predictive differences among fixed m values are partition-sensitive.** The inner-selected analysis is the "
    "only unbiased estimate of choosing m adaptively in this dataset.\n"
    "4. **The manuscript should not state that m=4 was retained because it ranked highest in the full cohort.** "
    "It may be retained for visualization as an intermediate 24-state representation only if described as exploratory."
))

### Save execution manifest

In [ ]:
manifest = {
    "project_root": str(PROJECT_ROOT),
    "dataset_root": str(DATASET_ROOT),
    "output": str(OUT),
    "seed": SEED,
    "n_repeats": N_REPEATS,
    "outer_folds": OUTER_FOLDS,
    "inner_folds": INNER_FOLDS,
    "fixed_epochs": N_FIXED_EPOCHS,
    "embedding_dimensions": list(M_VALUES),
    "k_features": K_FEATURES,
    "n_trees": N_TREES,
    "python": sys.version,
    "package_versions": {
        "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
    },
    "outputs": sorted(str(path.relative_to(OUT)) for path in OUT.rglob("*") if path.is_file()),
}
(OUT / "notebook_08_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
display(manifest)